# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

M. Benisty  ->  M. Benisty  |  ['M. Benisty']
H. Jiang  ->  H. Jiang  |  ['H. Jiang']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
E. Matthews  ->  E. Matthews  |  ['E. Matthews']
M. Bergemann  ->  M. Bergemann  |  ['M. Bergemann']
N. Storm  ->  N. Storm  |  ['N. Storm']
G. Guiglion  ->  G. Guiglion  |  ['G. Guiglion']
A. Dimoff  ->  A. Dimoff  |  ['A. Dimoff']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
J. Li  ->  J. Li  |  ['J. Li']
H. Jiang  ->  H. Jiang  |  ['H. Jiang']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
K. A  ->  K. A. Kahle  |  ['K. A']
M. Zhang  ->  M. Zhang  |  ['M. Zhang']


M. Benisty  ->  M. Benisty  |  ['M. Benisty']
Y. Reinarz  ->  Y. Reinarz  |  ['Y. Reinarz']
Arxiv has 100 new papers today
          12 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/12 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.10103


extracting tarball to tmp_2611.10103... done.
Retrieving document from  https://arxiv.org/e-print/2611.10129


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.10103/ms.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'acro_commands' from 'tmp_2611.10103/acro_commands.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


extracting tarball to tmp_2611.10129... done.


Issues with the citations
repeated bibliography entry: Dulick2003
Retrieving document from  https://arxiv.org/e-print/2611.10132


... (9 outputs left out)

M. Benisty  ->  M. Benisty  |  ['M. Benisty']


Retrieving document from  https://arxiv.org/e-print/2611.12581
extracting tarball to tmp_2611.12581... done.


Y. Reinarz  ->  Y. Reinarz  |  ['Y. Reinarz']


Found 119 bibliographic references in tmp_2611.12581/aa60762-26.bbl.
Issues with the citations
syntax error in line 718: '=' expected


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.10129-b31b1b.svg)](https://arxiv.org/abs/2611.10129) | **The deuterium-rich atmosphere of a mature super-Jupiter**  |
|| J. W. Xuan, et al. -- incl., <mark>Y. Zhang</mark>, <mark>E. Matthews</mark> |
|*Appeared on*| *2026-11-10*|
|*Comments*| *Under review*|
|**Abstract**|            The deuterium-to-hydrogen ratio (D/H) has provided key insights into the origin and evolution of the Solar System. Using the James Webb Space Telescope, we detected deuterated methane at $48\sigma$ in a mature giant exoplanet, $\epsilon$ Ind Ab. We measure the planet's D/H ratio to be $(4.4 \pm 0.7) \times 10^{-5}$, which is similar to those of Uranus and Neptune but about twice those of Jupiter, Saturn, and the protosolar nebula. $\epsilon$ Ind Ab's heavy elements (C, O, N, and S) are uniformly enriched to a level similar to Jupiter's. These measurements require $\epsilon$ Ind Ab to have accreted solids with $\mathrm{D/H} \gtrsim 1\%$, at least 30 times more deuterium-rich than typical Solar System comets. Such D/H values are found in the interstellar comet 3I/ATLAS and in organic molecules in star-forming regions, indicating that $\epsilon$ Ind Ab accreted pristine solids formed in a very cold environment.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.10132-b31b1b.svg)](https://arxiv.org/abs/2611.10132) | **Classical Cepheids in the 4MOST era I: Methodology for 4MOST variable stars and NLTE abundance gradients in the Milky Way**  |
|| R. Albarracín, et al. -- incl., <mark>M. Bergemann</mark>, <mark>N. Storm</mark>, <mark>G. Guiglion</mark>, <mark>A. Dimoff</mark> |
|*Appeared on*| *2026-11-10*|
|*Comments*| *24 pages, 19 figures, Accepted for publication by A&A*|
|**Abstract**|            Classical Cepheids (CCs) are key distance indicators and tracers of young stellar populations in the Milky Way. Their period-luminosity-metallicity (PLZ) relation also makes them calibrators of the Hubble constant $H_0$. Despite their importance, chemical composition, especially non-LTE (NLTE) effects, remains a significant source of systematics in PLZ applications. In preparation for the large CCs sample expected from the 4MIDABLE-HR (4MOST Consortium Survey 4: Milky Way Disc and Bulge High-Resolution) survey, we present a method for deriving Cepheid atmospheric parameters and chemical abundances in NLTE. Using the Payne artificial neural network (ANN) trained on NLTE spectra, we derive atmospheric parameters and abundances for 17 elements: O, Na, Mg, Al, Si, Ca, Ti, Mn, Fe, Co, Ni, Sr, Y, Ba, Eu in NLTE and C and Cr in LTE. We apply the new method to archival spectra of Cepheids and derive radial chemical gradients in the Milky Way. We derive robust NLTE stellar parameters and abundances for Cepheids spanning metallicities from $\sim -0.87$ to $\sim +0.35$ dex. For stars with multi-epoch spectra, we find that NLTE abundances are consistent across the pulsation cycle, whereas LTE values show larger dispersions and biases. The resulting abundance distributions and gradients align well with Galactic disk expectations from chemical evolution models. We confirm that the gradients flatten in the outer disk beyond $R_{\rm GC} \sim 15\,\mathrm{kpc}$ and provide the first NLTE gradients for Co, Ni, Y, and Eu. We show that our NLTE astrophysical analysis yields robust solutions for atmospheric parameters and chemical abundances across spectra at the resolving power of 4MOST-HR. This enables large, homogeneous studies with upcoming surveys such as 4MOST 4MIDABLE-HR and provides a solid basis for tightening constraints on Galactic chemical evolution models and Cepheid distance-scale work.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.12504-b31b1b.svg)](https://arxiv.org/abs/2611.12504) | **ALMA detection of continuum emission from GQ Lup B**  |
|| P. Pirola, et al. -- incl., <mark>M. Benisty</mark> |
|*Appeared on*| *2026-11-10*|
|*Comments*| *12 pages, 11 figures, 2 tables including appendices. Accepted for publication in A&A*|
|**Abstract**|            GQ Lup B is a massive (10-40 M$_{\text{Jup}}$) companion orbiting the young T-Tauri star GQ Lup A at a projected distance of $\sim$110 au (0.7''). It is outside the circumprimary disk in the submm continuum, but its ${\sim}\,63^\circ$ inclined orbit intersects the primary's gas disk twice per period. We report a 5.8$\sigma$ detection of continuum emission at the location of GQ Lup B, obtained with the Atacama Large Millimeter/submillimeter Array (ALMA) at 0.89 mm (band 7). We associate this emission with a dusty disk around the companion, already detected by VLT/NACO and JWST/MIRI in the mid-Infrared. To date, this is the third disk observed at mm wavelengths around a companion spanning the brown dwarf and planetary mass regimes, following PDS 70 c, a gas giant embedded in the parent circumstellar disk, and SR 12 c, a planetary-mass companion at 1200 au from the primary. We retrieve a point source with peak intensity $54.5\pm9.4\,\mu\rm Jy\, beam^{-1}$, co-spatial with GRAVITY astrometry. The emission is spatially unresolved, placing an upper limit on the source radius of 2.3 au (from a synthesized beam of $\sim\,29\times25$ mas). For comparison, this is smaller than 1/3 of the Hill's radius (3-5 au for GQ Lup B, depending on the assumed mass), the theoretical upper limit for circum-planetary disks. Assuming optically thin thermal emission from mm-size grains in a disk-like structure, we infer a mass of 0.0032 M$_\oplus$, while the optically thick limit yields an emission radius of 0.2 au. Assuming a gas to dust ratio of 100, our results indicate a very low disk mass to accretion rate ratio, of the order of $\sim10^4$ years, analogously to the highest accretors in the stellar mass regime. In the case of GQ Lup B, this can be explained by its periodic crossings of the circumprimary disk, which could deliver a substantial amount of gas to the companion's disk over a long timescale.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.12581-b31b1b.svg)](https://arxiv.org/abs/2611.12581) | **The compositional link between rocky exoplanets and their host stars is consistent with Earth-like devolatilization**  |
|| <mark>Y. Reinarz</mark>, L. Acuña-Aguirre, H. S. Wang |
|*Appeared on*| *2026-11-10*|
|*Comments*| *15 pages, 9 figures, 7 tables. Accepted for publication in Astronomy & Astrophysics*|
|**Abstract**|            Previous work found a higher bulk iron-to-silicon (Fe/Si) ratio in rocky exoplanets relative to their host star, traditionally explained by iron enrichment. An alternative explanation is devolatilization of the planetary abundance ratios with respect to the host star, where volatile elements such as O and S are progressively depleted compared to refractory elements such as Fe and Mg, as in Earth. We explore whether these elemental molar ratios are consistent with a devolatilization trend. We revisit the core mass fraction of 14 rocky exoplanets with masses between 0.5 and 10 Earth masses whose host stars have spectroscopic abundances of Fe, Si, Mg, O, and S in the literature. We use interior structure retrievals with two independent forward models, the Marseille Super-Earth Interior model and pyExoInt, to estimate compositional parameters, including elemental molar ratios, from mass and radius measurements. We find that 8 out of 14 exoplanets in our sample present an Earth-like devolatilization trend, while only one planet shows a 1:1 relationship between planetary and stellar abundances. This suggests that devolatilization offers a plausible explanation for the observed non-1:1 relation. The two interior models agree on the devolatilization trend, demonstrating that our derived trends are robust. The main limiting factor in constraining refractory compositions of rocky planets relative to their host star is the small sample with constrained stellar abundances and the lack of variety in orbital distances. The PLATO mission will help by extending the sample of characterized rocky planets to orbital periods of up to one year. The variety in refractory elemental ratio trends may indicate a wide diversity in bulk composition, outgassing and other mantle properties relevant for habitability.         |

## Failed papers

... (4 outputs left out)


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.11107-b31b1b.svg)](https://arxiv.org/abs/2611.11107) | **Probing faint radio activity from cores to filaments in two $z=2.24$ protoclusters with uGMRT**  |
|| <mark>Y. Zhang</mark>, et al. |
|*Appeared on*| *2026-11-10*|
|*Comments*| *A&A in press. 11 pages, 6 figures, 1 table*|
|**Abstract**|            We present deep 1.26 GHz radio continuum observations of the two $z=2.24$ protoclusters BOSS1244 and BOSS1542 with the upgraded Giant Metrewave Radio Telescope (uGMRT) to investigate their members' radio emission and its possible environmental dependence. Among 187 and 105 H$\alpha$ emitter (HAE) members in BOSS1244 and BOSS1542, we identify ten and two robust radio counterparts, respectively, together with five potential counterparts from forced photometry. Two HAEs and the massive quiescent galaxy QG1 in BOSS1244 show strong radio excesses relative to their H$\alpha$-based star formation rates (SFRs) and are identified as strong radio-excess AGN candidates. Median stacking of the remaining 173 and 102 radio-undetected HAE members yields positive signals at $2.8\sigma$ and $3.2\sigma$ in BOSS1244 and BOSS1542, respectively. The stacked radio luminosities are consistent with those expected from the median H$\alpha$-based SFRs, indicating that star formation can account for the average faint radio emission. After excluding the robust radio counterparts, we divided the HAE members by the projected distance from QG1 in BOSS1244 and from the filament spine in BOSS1542. The largest stacked radio luminosities are found toward the denser regions in both systems, but the differences are not statistically significant, and stellar mass shows no significant dependence on environmental distance. These results identify strong radio-excess AGN candidates among the individually detected members, while the faint HAE population is on average consistent with radio emission from star formation. The similar environmental ordering is suggestive but remains tentative. Deeper radio observations at multiple frequencies and higher angular resolution are required to determine whether this apparent trend reflects weak AGN activity or variations in radio emission associated with star formation.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.11617-b31b1b.svg)](https://arxiv.org/abs/2611.11617) | **Unveiling a Concentration of Hard X-Ray Point Sources in the Nuclear Stellar Disk with XMM-Newton**  |
|| S. Mandel, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-10*|
|*Comments*| *Submitted to ApJ; comments welcome*|
|**Abstract**|            The Galactic Center is home to the largest known concentration of X-ray sources ever detected in our Galaxy, including a sizable population of accreting compact objects. Utilizing data from the XMM-Newton Galactic Plane Heritage Survey of the central $20^{\circ}\times2^{\circ}$ of the Milky Way, we investigate the varying source populations in the inner regions of the Galaxy. We identify for the first time a concentration of extremely hard X-ray sources within $|l|<1.3^{\circ}$ and $|b|<0.3^{\circ}$, a region that overlaps the nuclear stellar disk (NSD). We determine that this hardness is intrinsic to the sources and not a manifestation of high line-of-sight column density ($N_{\rm H}$). Unlike the "soft" X-ray sources, this population does not trace the stellar mass distribution, indicating an overabundance compared to the underlying stellar population. We simulate spectra for a variety of source types to model their X-ray hardness and compare their distributions to those we observed in our survey data. We find that a significant subset of the Galactic Center X-ray population may comprise intermediate polars (IPs) with massive white dwarfs, possibly formed through dynamical processes. Our study provides novel observational insights into the distribution of stellar remnants and other X-ray sources in the vicinity of our supermassive black hole, Sgr A*.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.12135-b31b1b.svg)](https://arxiv.org/abs/2611.12135) | **Particle Swarm Optimization Strategies for Lensed Gravitational Waves: Parameter Estimation, Source Classification, and Accelerated Sampling**  |
|| W. Yu, <mark>M. Zhang</mark>, X. Zou, X. Guo |
|*Appeared on*| *2026-11-10*|
|*Comments*| **|
|**Abstract**|            The detection and parameter estimation of lensed gravitational wave (GW) signals are crucial for astrophysics and cosmology. Strong lensing introduces extra parameters, rendering the search high-dimensional, multi-modal, and strongly nonlinear. We apply Particle Swarm Optimization (PSO) to the search, parameter estimation, and classification of lensed compact binary coalescence GWs within a four-detector coherent network. Using the restricted 2-PN waveform as a template and a model-independent double-image lens parameterization, we assess PSO on both model-matched 2-PN signals and template-mismatched IMRPhenomD signals. Results show effective recovery of chirp mass, mass ratio, sky location, and lens parameters in the 2-PN case, with mass biases appearing under waveform mismatch. A three-class classification pipeline combining 12 PSO runs, majority voting, and a flux ratio threshold achieves 86.19% accuracy in distinguishing lensed GWs, unlensed GWs, and noise. We further propose PSO_GAUSS_MIX method, which constructs a Gaussian mixture prior from PSO estimates for nested sampling, yielding speedups of 1.38x in the restricted 2-PN cases while preserving posterior accuracy. These results establish PSO as an efficient optimizer for lensed GW searches and an effective prior constructor for Bayesian parameter estimation.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.12046-b31b1b.svg)](https://arxiv.org/abs/2611.12046) | **Could globular clusters in the vicinity of the galactic centre have contributed stars to the nuclear star cluster in the past?**  |
|| K. M., et al. -- incl., <mark>K. A</mark> |
|*Appeared on*| *2026-11-10*|
|*Comments*| *10 pages, 5 figures, 2 Tables*|
|**Abstract**|            The results of direct N-body simulations of three Milky Way bulge globular clusters - Terzan 2, Terzan 4, and Terzan 5 - evolving in a time-varying galactic potential derived from the IllustrisTNG-100 cosmological database over 8 Gyr are presented. Incorporating full stellar evolution and compact remnant formation, we quantify the probability and efficiency of stellar accretion onto the Galactic nuclear star cluster (NSC) during close pericentre passages of these three clusters. To parametrize the effect of dynamical friction in the dense central environment, we introduce a velocity reduction factor (Vred) ranging from 1.00 to 0.10. It was found that all three clusters contribute stars to the NSC even in the absence of velocity reduction, with a combined bound stellar count of 730 stars and a total accreted mass of 278 Msol at Vred = 1.00, growing to 5420 stars and 1175 Msol at Vred = 0.10. Terzan 4 is the dominant contributor across all velocity reduction factors, reflecting its more frequent and closer pericentre passages near the Galactic centre. The mass distribution of accreted stars reveals a bimodal structure: a low-mass component below 1 Msol whose capture is independent of Vred, and a prominent high-mass tail extending to 20 Msol corresponding to neutron stars and black holes, which grows with decreasing Vred. Notably, Terzan 4 is the only cluster for which white dwarfs are also captured by the NSC at every Vred value. Our results suggest that the Terzan family represents a more significant source of compact remnants for the Galactic NSC than previously studied clusters, while confirming that tidal stripping of surviving globular clusters alone cannot account for the total NSC mass.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error not a gzip file</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.10129/./epsind_detection.png', 'tmp_2611.10129/./epsind_ch3d.png', 'tmp_2611.10129/./epsindi_dh.png']
copying  tmp_2611.10129/./epsind_detection.png to _build/html/
copying  tmp_2611.10129/./epsind_ch3d.png to _build/html/
copying  tmp_2611.10129/./epsindi_dh.png to _build/html/
exported in  _build/html/2611.10129.md
    + _build/html/tmp_2611.10129/./epsind_detection.png
    + _build/html/tmp_2611.10129/./epsind_ch3d.png
    + _build/html/tmp_2611.10129/./epsindi_dh.png
found figures ['tmp_2611.10132/./figures/Calib_VariableStar_multiphase_R-Tra.png', 'tmp_2611.10132/./figures/Callibrating_cepheids_params_two_panel_vertical.png', 'tmp_2611.10132/./figures/abundances_2x3.png']
copying  tmp_2611.10132/./figures/Calib_VariableStar_multiphase_R-Tra.png to _build/html/
copying  tmp_2611.10132/./figures/Callibrating_cepheids_params_two_panel_vertical.png to _build/html/
copying  tmp_2611.10132/./figures/abundances_2x3.png to _build/html/
exported in  _build/html/261

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\pcc}{ cm^{-3}}$
$\newcommand{\Rstar}{\ensuremath{R_{\star}}\xspace}$
$\newcommand{\msini}{\ensuremath{m \sin I}\xspace}$
$\newcommand{\ms}{\ensuremath{\mathrm{m} \mathrm{s}^{-1}}\xspace}$
$\newcommand{\masyr}{\ensuremath{\mathrm{mas} \mathrm{yr}^{-1}}\xspace}$
$\newcommand{\Me}{\ensuremath{M_{\oplus}}\xspace}$
$\newcommand{\Rj}{\ensuremath{R_{\rm{Jup}}}\xspace}$
$\newcommand{\Mj}{\ensuremath{M_{\rm{Jup}}}\xspace}$
$\newcommand{\kms}{km~s^{-1}\xspace}$
$\newcommand{\Rsun}{\ensuremath{R_{\odot}}\xspace}$
$\newcommand{\Msun}{\ensuremath{M_{\odot}}\xspace}$
$\newcommand{\Lsun}{L_\odot}$
$\newcommand{\Teff}{\ensuremath{T_\mathrm{eff}}\xspace}$
$\newcommand{\logg}{\ensuremath{\log{g}}\xspace}$
$\newcommand{\lbollsun}{\ensuremath{\log(L_\mathrm{bol}/\Lsun)}}$
$\newcommand\refname{References and Notes}$
$\newcommand{\fnum@figure}{\textbf{Figure \thefigure}}$
$\newcommand{\fnum@table}{\textbf{Table \thetable}}$
$\newcommand{\Re}{\ensuremath{R_{\oplus}}\xspace}$
$\newcommand{\thefigure}{S\arabic{figure}}$
$\newcommand{\thetable}{S\arabic{table}}$
$\newcommand{\theequation}{S\arabic{equation}}$
$\newcommand{\thepage}{S\arabic{page}}$
$\newcommand\scititle{$
$	The deuterium-rich atmosphere of a mature super-Jupiter$
$}$</div>



<div id="title">

# $\bfseries$ $\boldmath$ $\scititle$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.10129-b31b1b.svg)](https://arxiv.org/abs/2611.10129)<mark>Appeared on: 2026-11-10</mark> -  _Under review_

</div>
<div id="authors">

J. W. Xuan, et al. -- incl., <mark>Y. Zhang</mark>, <mark>E. Matthews</mark>

</div>
<div id="abstract">

**Abstract:** $\bfseries$ $\boldmath$ The deuterium-to-hydrogen ratio (D/H) has provided key insights into the origin and evolution of the Solar System. Using the James Webb Space Telescope, we detected deuterated methane at $48\sigma$ in a mature giant exoplanet, $\epsilon$ Ind Ab. We measure the planet's D/H ratio to be $(4.4\pm0.7)\times10^{-5}$ , which is similar to those of Uranus and Neptune but about twice those of Jupiter, Saturn, and the protosolar nebula. $\epsilon$ Ind Ab's heavy elements (C, O, N, and S) are uniformly enriched to a level similar to Jupiter's. These measurements require $\epsilon$ Ind Ab to have accreted solids with $\rm D/H\gtrsim1\%$ , at least 30 times more deuterium-rich than typical Solar System comets. Such D/H values are found in the interstellar comet 3I/ATLAS and in organic molecules in star-forming regions, indicating that $\epsilon$ Ind Ab accreted pristine solids formed in a very cold environment.

</div>

<div id="div_fig1">

<img src="tmp_2611.10129/./epsind_detection.png" alt="Fig1" width="100%"/>

**Figure 1. -** **Detection of $\epsilon$ Ind Ab with moderate-resolution mode of JWST/NIRSpec IFU in the $3-5 \mu$m spectral range.****a,** The median spectral cube for one of the two NIRSpec rolls from the standard JWST pipeline before starlight subtraction. Here, the stellar diffraction pattern dominates the image. The host star $\epsilon$ Ind A is just outside the field of view of the IFU, and the red arrow marks its direction. **b,** S/N map for $\epsilon$ Ind Ab combining the two NIRSpec rolls and both NIRSpec detectors. Summed over all wavelength channels, the planet is detected with a S/N of approximately 600.
 (*fig:detection*)

</div>
<div id="div_fig2">

<img src="tmp_2611.10129/./epsind_ch3d.png" alt="Fig2" width="100%"/>

**Figure 2. -** **Detection of $CH_3$D in $\epsilon$ Ind Ab.****a,** A segment of the NIRSpec spectrum (black). The best-fit full model including all species is in blue, and the best-fit model from a separate retrieval that excludes $CH_3$D opacity is in orange. The difference between the blue and orange models is due to $CH_3$D; we mark locations of $CH_3$D lines with red dashed lines and arrows. **b,** The residuals after subtracting the best-fit model without $CH_3$D from the data (black), plotted against the $CH_3$D-only model (red). A cross-correlation analysis between the residuals and $CH_3$D model indicates a detection $\rm S/N\approx48$ for $CH_3$D in $\epsilon$ Ind Ab's atmosphere. Equivalent plots to panel b for other species are shown in figure \ref{fig:other_detections}.
 (*fig:spectrum*)

</div>
<div id="div_fig3">

<img src="tmp_2611.10129/./epsindi_dh.png" alt="Fig3" width="100%"/>

**Figure 3. -** **D/H measurements in various astrophysical environments, including planets, comets, circumstellar disks, and protostars.** The black markers indicate estimates of bulk D/H, either measured directly from atomic or molecular hydrogen or inferred from other molecules with appropriate corrections ($CH_3$D/$CH_4$ for $\epsilon$ Ind Ab and WISE 0855). The other colors denote D/H in specific molecular tracers, and do not represent bulk D/H: $H_2$O (blue), $CH_4$(teal), HCN (yellow), gas-phase deuterated methanol, $CH_2$DOH (magenta), and refractory organics (brown). Data sources: primordial value \cite{Cooke2018}; WISE 0855 \cite{Rowland2024}; local ISM, Solar System planets, most Solar System comets, and protostars, from a published compilation \cite{Nomura2023}; 3I/ATLAS \cite{Cordiner2026,2026arXiv260320445R}; 67P/Churyumov--Gerasimenko, using updated values \cite{Mandt2024,Paquette2021}; V883 Ori \cite{Tobin2023}; HCN in other protoplanetary disks \cite{Huang2017}.
 (*fig:dtoh*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.10129"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\gradientpm}[2]{\makebox[3.3em][r]{#1}\ensuremath{ \pm }\makebox[3.3em][l]{#2}}$
$\newcommand{\gradientci}[3]{\makebox[3.3em][r]{#1}\ensuremath{ _{-#2}^{+#3}}}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\hnot}{\mathrm{H}_{0}}$
$\newcommand{\hnotsp}{\mathrm{H}_{0}  }$
$\newcommand{\angstrom}{\mbox{\normalfontÅ}}$
$\newcommand{\chisq}{\chi^{2}}$
$\newcommand{\ironab}{\left[\mathrm{Fe} / \mathrm{H}\right] }$
$\newcommand{\teff}{T_{\rm eff}  }$
$\newcommand{\Gbp}{\mathrm{G}_{\mathrm{BP}}  }$
$\newcommand{\Grp}{\mathrm{G}_{\mathrm{RP}}  }$
$\newcommand{\Gbpnosp}{\mathrm{G}_{\mathrm{BP}}}$
$\newcommand{\Grpnosp}{\mathrm{G}_{\mathrm{RP}}}$
$\newcommand{\Rsun}{R_{\odot} }$
$\newcommand{\logg}{\log{g} }$
$\newcommand{\vmic}{\mathrm{V}_{\mathrm{mic}}  }$
$\newcommand{\vbrd}{\rm{V}_{\rm{brd}}  }$
$\newcommand{\teffnospace}{T_{\rm eff}}$
$\newcommand{\loggnospace}{\log{g}}$
$\newcommand{\vmicnospace}{\mathrm{V}_{\mathrm{mic}}}$
$\newcommand{\vbrdnospace}{\rm{V}_{\rm{brd}}}$
$\newcommand{\arraystretch}{0.90}$
$\newcommand{\arraystretch}{1.00}$
$\newcommand{\arraystretch}{0.90}$
$\newcommand{\arraystretch}{0.90}$</div>



<div id="title">

# Classical Cepheids in the 4MOST era: I. Methodology for 4MOST variable stars and NLTE abundance gradients in the Milky Way.

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.10132-b31b1b.svg)](https://arxiv.org/abs/2611.10132)<mark>Appeared on: 2026-11-10</mark> -  _24 pages, 19 figures, Accepted for publication by A&A_

</div>
<div id="authors">

R. Albarracín, et al. -- incl., <mark>M. Bergemann</mark>, <mark>N. Storm</mark>, <mark>G. Guiglion</mark>, <mark>A. Dimoff</mark>

</div>
<div id="abstract">

**Abstract:** Classical Cepheids (CCs) are key distance indicators and tracers of young stellar populations in the Milky Way. Their period-luminosity-metallicity (PLZ) relation also makes them calibrators of the Hubble constant $H_0$ . Despite their importance, chemical composition, especially non-LTE (NLTE) effects, remains a significant source of systematics in PLZ applications. In preparation for the large sample of CCs expected from the 4MIDABLE-HR (4MOST Consortium Survey 4: Milky Way Disc and Bulge High-Resolution) survey, we present a method for deriving Cepheid atmospheric parameters and chemical abundances in NLTE. Using the Payne artificial neural network (ANN) trained on NLTE spectra, we derive atmospheric parameters and abundances for 17 elements: O, Na, Mg, Al, Si, Ca, Ti, Mn, Fe, Co, Ni, Sr, Y, Ba, Eu in NLTE and C and Cr in LTE. We apply the new method to archival spectra of Cepheids and derive radial chemical gradients in the Milky Way. We derive robust NLTE stellar parameters and abundances for Cepheids spanning a broad metallicity range, from $\sim -0.87   \rm{dex}$ to $\sim +0.35   \rm{dex}$ . For stars with multi-epoch spectra, we find that NLTE abundances are consistent across the pulsation cycle, whereas LTE values show larger dispersions and biases. The resulting abundance distributions and gradients align well with Galactic disk expectations from chemical evolution models. We confirm that the gradients flatten in the outer disk beyond $R_{\rm GC} \sim 15 \mathrm{kpc}$ and provide the first NLTE gradients for Co, Ni, Y, and Eu. We show that our NLTE astrophysical analysis yields robust solutions for atmospheric parameters and chemical abundances across spectra at the resolving power of 4MOST-HR. This enables large, homogeneous studies with upcoming surveys such as 4MOST 4MIDABLE-HR and provides a solid basis for tightening constraints on Galactic chemical evolution models and Cepheid distance-scale work.

</div>

<div id="div_fig1">

<img src="tmp_2611.10132/./figures/Calib_VariableStar_multiphase_R-Tra.png" alt="Fig16" width="100%"/>

**Figure 16. -** Multi-epoch spectra of CC R TrA in NLTE and LTE. Top: Portion of the spectra in the vicinity of the $\rm{H}_{\alpha}$ line; the colors indicate the normalized spectra at different pulsation phases. Lines present in our line list are overplotted. Bottom left: Line profiles of key diagnostic features across phase, with the observed spectra shown in black and the best-fitting model in green. Bottom right: Phase dependent abundances for a selection of elements. Light green points on the left represent NLTE abundances for the phases available, while red points on the right correspond to LTE abundances. The NLTE results demonstrate stable solutions in spite of the stellar variability. Only measurements with both stellar parameters and abundances lying within the grid limits are displayed. Dashed lines indicate phase-averaged values, and the shaded regions denote the standard deviation of abundances in NLTE (green) and LTE (red). (*fig:CalibMuliphase*)

</div>
<div id="div_fig2">

<img src="tmp_2611.10132/./figures/Callibrating_cepheids_params_two_panel_vertical.png" alt="Fig3" width="100%"/>

**Figure 3. -** $\teff$nospace -$\logg$nospace  diagram (top) and $\teff$nospace -luminosity diagram (bottom) of selected CCs with multiphase observations. Squares show the parameters derived from individual spectra, solid lines represent the Fourier fits to the phase progression of $\teff$nospace , $\logg$nospace , and luminosity. The gray region shows the instability strip from [Anderson, et. al (2016)](https://ui.adsabs.harvard.edu/abs/2016A&A...591A...8A) for a solar-metallicity star with a rotation rate equal to half the critical angular velocity, as defined by [Georgy, et. al (2013)](https://ui.adsabs.harvard.edu/abs/2013A&A...553A..24G). Filled circles show the average positions of the stars in the HR diagram across phase. Crosses show the typical uncertainties in $\teff$nospace , $\logg$nospace , and luminosity. (*fig:multiphase_5_show*)

</div>
<div id="div_fig3">

<img src="tmp_2611.10132/./figures/abundances_2x3.png" alt="Fig4" width="100%"/>

**Figure 4. -** Abundance histograms for the 6 CCs shown in Fig. \ref{fig:multiphase_5_show}, obtained in NLTE in this work (solid) and reported by [Nunnari, et. al (2026)](https://ui.adsabs.harvard.edu/abs/2026A&A...708A..17N)(hatched) for the spectra common to both studies. Each panel shows the NLTE abundance distribution of an individual element, shown in the top left. (*fig:multiphase_5_abundances*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.10132"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\tnote}[1]{\textnormal{ [\textit{#1}]}}$
$\newcommand\TPT@dopara{\let\item\TPT@paraitem\par\noindent}$
$\newcommand\TPT@paraitem{[}$</div>



<div id="title">

# ALMA detection of continuum emission from GQ Lup B

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.12504-b31b1b.svg)](https://arxiv.org/abs/2611.12504)<mark>Appeared on: 2026-11-10</mark> -  _12 pages, 11 figures, 2 tables including appendices. Accepted for publication in A&A_

</div>
<div id="authors">

P. Pirola, et al. -- incl., <mark>M. Benisty</mark>

</div>
<div id="abstract">

**Abstract:** GQ Lup B is a massive (10-40 M $_{\text{Jup}}$ ) companion orbiting the young T-Tauri star GQ Lup A at a projected distance of $\sim$ 110 au (0.7"). It is outside the circumprimary ${disk in the submm continuum}$ , but its ${\sim} 63^\circ$ inclined orbit intersects the primary's gas disk twice per period.We report a 5.8 $\sigma$ detection of continuum emission at the location of GQ Lup B, obtained with the Atacama Large Millimeter/submillimeter Array (ALMA) at 0.89 mm (band 7). We associate this emission with a dusty disk around the companion, already detected by VLT/NACO and JWST/MIRI in the mid-Infrared. To date, this is the third disk observed at mm wavelengths around ${a companion spanning the brown dwarf and planetary mass regimes}$ , following PDS 70 c, a gas giant embedded in the parent circumstellar disk, and SR 12 c, a planetary-mass companion at 1200 au from the primary.We retrieve a point source with peak intensity $54.5\pm9.4 \mu\rm Jy  beam^{-1}$ , co-spatial with GRAVITY astrometry.The emission is spatially unresolved, ${placing}$ an upper limit on the source radius of 2.3 au ${(from a synthesized beam of $\sim 29\times25$ mas). For comparison, this is smaller than 1/3 of the Hill's radius (3-5 au for GQ Lup B, depending on the assumed mass), the theoretical upper limit for circum-planetary disks.}$ Assuming optically thin thermal emission from mm-size grains ${in a disk-like structure}$ , we infer a mass of 0.0032 M $_\oplus$ , while the optically thick limit yields an emission radius of 0.2 au. ${Assuming a gas to dust ratio of 100, our results indicate a very low disk mass to accretion rate ratio, of the order of $\sim10^4$ years, analogously to the highest accretors in the stellar mass regime. In the case of GQ Lup B, this can be explained by its periodic crossings of the circumprimary disk, which could deliver a substantial amount of gas to the companion's disk over a long timescale.}$

</div>

<div id="div_fig1">

<img src="tmp_2611.12504/./continuum_and_scatteredlight_contours_zoom.png" alt="Fig9" width="100%"/>

**Figure 9. -** _ Left:_  scattered light image of GQ Lup taken with VLT/SPHERE  ([Holstein, et. al 2021]()) . A white contour (at 40\% of the peak of the image) serves as a reference for the other panels. The single spiral arm visible in the southern part of the image is attributed to the companion, which appears in the right side of the image. Although useful for spatial comparison, the apparent polarized flux of GQ Lup B is caused by the subtraction of the stellar component, as explained in the original paper  ([Holstein, et. al 2021]()) .
    _ Center:_ saturated version of the continuum image from Fig. \ref{fig:continuum_fullscale}, where the color scale has been limited to a range of 0 to $50 \mu\rm Jy \rm  beam ^{-1}$. The dashed gray square, with sides of $0.5"$, is centered on GQ Lup B.
    _ Right:_ zoomed-in view of the same region as in the central panel: the red X marks the companion's position from GRAVITY {on 2023/3/8  ([Venkatesan, et. al 2025]()) , while the white contour} represents the scattered light contour. (*fig:continuumvsscattered*)

</div>
<div id="div_fig2">

<img src="tmp_2611.12504/./Mdust_vs_Mdot_cmap.png" alt="Fig11" width="100%"/>

**Figure 11. -** Accretion rates for disks around stars (circles) and planetary-mass objects (squares, the same discussed in Fig. \ref{fig:fvsmass}), against dust mass estimates (Eq \ref{eq:Mthin}, T=20 K) multiplied by 100 to account for the gas mass. Triangles represent upper limits. The color scale here accounts for the mass of the disk host, while dashed gray lines represent depletion timescales. (*fig:mdustvsmdot*)

</div>
<div id="div_fig3">

<img src="tmp_2611.12504/./continuum_image_fullscale.png" alt="Fig1" width="100%"/>

**Figure 1. -** 890$ \mu$m continuum image of GQ Lup (robust=2, beam $\sim 0.13 \times 0.11"$). The contours represent the 3,4, and 5$\sigma$ contours. Given the very high dynamic range of the image (${\sim} 2900$), we used an Asinh normalization of the color scale. (*fig:continuum_fullscale*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.12504"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$</div>



<div id="title">

# The compositional link between rocky exoplanets and their host stars is consistent with Earth-like devolatilization

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.12581-b31b1b.svg)](https://arxiv.org/abs/2611.12581)<mark>Appeared on: 2026-11-10</mark> -  _15 pages, 9 figures, 7 tables. Accepted for publication in Astronomy & Astrophysics_

</div>
<div id="authors">

<mark>Y. Reinarz</mark>, L. Acuña-Aguirre, H. S. Wang

</div>
<div id="abstract">

**Abstract:** Previous work found a higher bulk iron-to-silicon (Fe/Si) ratio in rocky exoplanets relative to their host star, traditionally explained by iron enrichment. An alternative explanation is devolatilization of the planetary abundance ratios with respect to the host star, where volatile elements such as oxygen and sulfur are progressively depleted compared to more refractory elements such as iron and magnesium, as in Earth. In this work, we explore whether these elemental molar ratios are consistent with a devolatilization trend. We revisit the estimation of the core mass fraction of 14 rocky exoplanets with masses between 0.5 and 10 Earth masses, focusing on planets where stellar abundances of Fe, Si, Mg, O, and S are available from stellar spectroscopy in the literature. We use interior structure retrievals with two independent forward models, the Marseille Super-Earth Interior model and pyExoInt, to estimate compositional parameters, including elemental molar ratios, from mass and radius measurements. We find that 8 out of 14 exoplanets in our sample present an Earth-like devolatilization trend, while only one planet shows a 1:1 relationship between the planetary and the stellar abundances. This suggests that devolatilization offers a plausible explanation for the observed non-1:1 relation. The two interior models are in good agreement on the devolatilization trend, demonstrating that our derived trends are robust. The main limiting factor in constraining refractory compositions between rocky planets and their host star is the small sample with constrained host stellar abundances and the lack of variety in orbital distances. The _PLATO_ mission will help solve this problem by extending the sample of characterized rocky planets to orbital periods of up to one year. The variety in refractory elemental ratio trends may be indicative of a wide diversity in bulk composition, outgassing and other mantle properties relevant for habitability.

</div>

<div id="div_fig1">

<img src="tmp_2611.12581/./figs/slope_plots.png" alt="Fig9" width="100%"/>

**Figure 9. -** Correlation between the devolatilization trend slope calculated with MSEI and key planetary parameters: orbital distance, equilibrium temperature, and planet mass. The blue circles represent planets whose host stars have abundances available for all the elements in our forward model, including S. The orange squares represent planets whose host star lacks a measured S abundance.  (*fig:corr*)

</div>
<div id="div_fig2">

<img src="tmp_2611.12581/./figs/sample.png" alt="Fig4" width="100%"/>

**Figure 4. -** Mass-radius diagram of the planets in our sample with respective errors. For comparison purposes we show MSEI forward models for CMF = 0.32 and CMF = 0.67, representing Super-Earths and Super-Mercuries respectively. The gray shaded region indicates the part of the mass-radius diagram corresponding to compositions less dense than the pure-mantle threshold, the lowest-density composition possible for a rocky planet. By selecting planets outside this gray area, we rule out planets whose density is inconsistent with a rocky interior without a gas layer (e.g. H$_{2}$O, H$_{2}$). (*fig:MR_diagram_Sample*)

</div>
<div id="div_fig3">

<img src="tmp_2611.12581/./figs/concentrical_model.png" alt="Fig1" width="100%"/>

**Figure 1. -** Schematic view of the interior model. (*fig:interior_model*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.12581"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

142  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

19  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

8  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
